In [2]:
# display warnings
import pandas as pd
from IPython.display import display, HTML
import warnings

# enable scrolling in large outputs
def show_scroll(df, height=400):
    display(HTML(f'<div style="height:{height}px; overflow:auto">{df.to_html()}</div>'))

pd.set_option("display.max_rows", 200)       # kaç satır göstersin
pd.set_option("display.max_colwidth", None)  # left/right kırpılmasın
warnings.filterwarnings("ignore", category=UserWarning)

In [3]:
import re 
from collections import Counter

CAPS_BLOCK = r"\b(?:[A-ZÇĞİÖŞÜ]{2,}\s+)+[A-ZÇĞİÖŞÜ]{2,}\b"
df = pd.read_csv(r"C:\Users\user\Desktop\chp_report\chp_report\data\corpus_main_processed.csv")

##########################
def remove_caps_blocks(text):
    return re.sub(CAPS_BLOCK, "", text)

df["text_nocaps"] = df["text_bert"].apply(remove_caps_blocks)

#########################
# inspect this later to complete the list
dont_touch = ["Dr.", "Prof.", "Doç.", "Hz.", "Sn.", "T.C.", "A.Ş.", "M.I.T",
              "Av.", "vs.", "vb.", "M.", "İ."]

def split_sentences(text):
    """"""
    for abbr in dont_touch:
        # \bdont_touch\. (e.g. \bDr\.)
        text = re.sub(r"\b" + re.escape(abbr), abbr.replace(".", "<DOT>"), text)
    pieces = re.split(r"[.?!…]+(?=\s|$)", text) # handles examples such as M.I.T.'ye, T.C.'nin 
    sentences = [piece.strip().replace("<DOT>", ".") for piece in pieces if piece.strip()]
    return sentences

df["sentences"] = df["text_nocaps"].apply(split_sentences)

#####################
def remove_short(sentences, min_words=4): # updated as 4, see the next cell
    return [s for s in sentences if len(s.split()) >= min_words]

df["sentences_long"] = df["sentences"].apply(remove_short)
    

#df[["text_bert", "n_sentences"]].head()
#show_scroll(df[["text_nocaps", "n_sentences"]])
df["n_sentences"] = df["sentences"].apply(len)
a = df["n_sentences"].sum()
df["n_sentences_long"] = df["sentences_long"].apply(len)
b = df["n_sentences_long"].sum()

print(f"before removing short sentences: {a}")
print(f"after removing short sentences: {b}")

# (df["n_sentences_long"] == 0).sum() = 1
sent_df = df.explode("sentences_long").dropna(subset=["sentences_long"])
len(sent_df)

sent_df = sent_df.reset_index().rename(columns={"index": "doc_id"})
sent_df = sent_df[["id", "date", "speaker", "venue", "sentences_long"]]
docs = sent_df["sentences_long"].tolist()
sent_df.columns


before removing short sentences: 602605
after removing short sentences: 515174


Index(['id', 'date', 'speaker', 'venue', 'sentences_long'], dtype='str')

In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

def turkish_lower(text):
    return text.replace("I", "ı").replace("İ", "i").lower()

with open(r"C:\Users\user\Desktop\chp_report\chp_report\data\stopwords_tr.txt", encoding="utf-8") as f:
    stopwords_tr = [turkish_lower(w.strip()) for w in f if w.strip()]

vectorizer = CountVectorizer(stop_words=stopwords_tr, min_df=10, preprocessor=turkish_lower)

model = SentenceTransformer('emrecan/bert-base-turkish-cased-mean-nli-stsb-tr', device="cuda")

embeddings = model.encode(docs, batch_size=64, show_progress_bar=True)
np.save("embeddings_emrecan.npy", embeddings)

hdbscan_model = HDBSCAN(min_cluster_size=150, min_samples=30, prediction_data=True)
topic_model = BERTopic(embedding_model=model, vectorizer_model=vectorizer, hdbscan_model=hdbscan_model)
topics, probs = topic_model.fit_transform(docs, embeddings)




Batches: 100%|██████████| 8050/8050 [06:17<00:00, 21.31it/s]


# stopwordleri konuları okuyarak güncelle, embeddings py'daki kelimeleri de al

In [6]:
a = topic_model.get_topic_info()
show_scroll(a)


,Topic,Count,Name,Representation,Representative_Docs
0,-1,253707,-1_türkiye_parti_diyor_siyasi,"[türkiye, parti, diyor, siyasi, erdoğan, başka, genel, chp, ak, ye]","[Ayrıca yayınlamış oldukları açıklamalar, beni arayarak bildirdikleri dayanışma duyguları ve takındıkları kurumsal tutumlarla DEVA Partisi’nin Sayın Genel Başkanı Ali Babacan’a, Demokrat Parti Genel Başkanı Gültekin Uysal’a, Türkiye İşçi Partisi Genel Başkanı Sayın Erkan Baş’a, EMEP’in Sayın Genel Başkanı Seyit Aslan’a, Gelecek Partisi Genel Başkanı Sayın Ahmet Davutoğlu’na, kurumsal açıklamalarıyla güç veren Saadet Partisi’ne ve Sol Parti’ye yürekten teşekkür ediyorum, Vicdan ne demektir; bilim insanları, ‘vicdan Allah’an kalbimizdeki sesidir’ der.10 Düşünün; bir ülkede bir İçişleri Bakanı çıkıyor diyor ki ‘Bir siyasetçiye her ay 10 bin dolar rüşvet veriliyor’, Ama Ak Parti'yi kapatacaklar]"
1,0,3188,0_emekli_emekliler_emekliye_emeklinin,"[emekli, emekliler, emekliye, emeklinin, emeklilere, ikramiye, emeklilerin, emeklileri, maaş, emekliyi]","[Bugün bir emekli değil Türkiye’deki her emekli, Bir emekli değil her emekli, Şimdi bir emekli değil her emekli]"
2,1,2738,1_ekrem_imamoğlu_başkan_beylikdüzü,"[ekrem, imamoğlu, başkan, beylikdüzü, nu, na, bey, beykoz, istanbul, cumhurbaşkanı]","[Ekrem İmamoğlu’nun tek korkuttuğu kişi; Tayyip Erdoğan, Ekrem İmamoğlu Cumhurbaşkanı adayımızdı, Zaten Ekrem İmamoğlu yok]"
3,2,2719,2_kadın_kadınlar_kadınların_kadına,"[kadın, kadınlar, kadınların, kadına, kadınlara, kadının, erkek, cinsiyet, kadınları, şiddet]","[Evet Cumhuriyet bir kadın devrimiydi, Özellikle kadınlar, kadın kardeşlerim size sesleniyorum, Cumhuriyet bir kadın devrimidir]"
4,3,2539,3_meclis_kanun_komisyonu_teklifi,"[meclis, kanun, komisyonu, teklifi, parlamentoya, komisyon, tbmm, parlamento, bütçe, meclisi]","[Değerli arkadaşlar vergi teklifi var Meclis’te, Bu komisyon kanun önerecek, Aslında o kanun tekliflerini de hazırlayanlar bürokratlardır]"
5,4,2442,4_demokrasiyi_demokrasi_demokrasinin_demokrasiye,"[demokrasiyi, demokrasi, demokrasinin, demokrasiye, demokrasiden, demokratik, savunuyoruz, ülkeye, getireceğiz, anlamda]","[Demokrasi bunun için var, Yani demokrasi kültürünüzü geliştireceksiniz, Ama birleşirsek demokrasi paydasında kaybetmeyiz]"
6,5,2250,5_saray_sarayda_sarayın_saraya,"[saray, sarayda, sarayın, saraya, saraydan, saraylarda, sarayı, uçan, yazlık, oturanlar]","[Biliyorsunuz bir saray sosyetesi var, Saray bana göre değil, Benim saray merakım yok]"
7,6,2248,6_yanlış_hata_eksiğimiz_ders,"[yanlış, hata, eksiğimiz, ders, yanlışımız, hatayı, kabahat, olabilir, düzelteceğiz, yanlışı]","[Yanlış olan da bu zaten, Bu doğru mudur, bu yanlış mıdır, Bu doğru mudur, yanlış mıdır]"
8,7,2080,7_sandığa_oy_giderken_sandığı,"[sandığa, oy, giderken, sandığı, kullanın, oyumuzu, vicdanınıza, elinizi, gideceğiz, sandık]","[Sandığa giderken elinizi vicdanınıza koyun ve oyunuzu öyle kullanın, Sandığa giderken elinizi vicdanınıza koyun ve oyunuzu öyle kullanın, Sandığa inanırız, sandığa güveniriz]"
9,8,1995,8_hakim_hakimler_yargı_dava,"[hakim, hakimler, yargı, dava, mahkeme, yargıtay, yargının, vicdani, karar, hukukun]","[Aslında tabii hakim kızdı biraz, Hakim hukukun üstünlüğü ve vicdani kanaatine göre karar verir, Hakim hukukun üstünlüğü ve vicdani kanaatine göre karar verir]"


In [75]:
#df.isna().sum() # 0 empty row

# any caps_block left?
#df["text_nocaps"].str.contains(r"[^.?!…\s]\s*" + CAPS_BLOCK).sum()

# test one example for the regex pattern
#test = " belki o zaman bilmiyorum… Hani konuşulanlar, Lazkiye'de onlara da bir otonom bölge sağlanıp, Esad’ın da güvenliğinin sağlandığı bir bölge teklif edilerek iktidarı devretmesi, kendisinin -belki bugün olduğu gibi ama- daha onurlu bir çıkışla ülkeyi terk edip Suriye’de bir geçiş hükümetine, demokrasiye doğru adım atmasını göze alsaydı bu hale gelmezdi, ülkesinden böyle kaçmazdı. Biz Suriye’ye sadece demokrasi telkin ettik. Ben Suriye’de Baas rejimi sürsün filan ister miyim? Ben niye isteyeyim yani? Ne hissedeceğim."
#re.split(r"[.?!…]+", test)

# inspection
#df["n_sentences"].describe()

# check dropped sentences (shorter than 5 words)
## update = 4 words still matter, lower the threshold to 4
dropped = df["sentences"].explode()
dropped = dropped[dropped.str.split().str.len() < 4] # 4
dropped.sample(30, random_state=4)

1404                                  Yok
785                                   Suç
169                   Tercihi kim yapacak
477                          İntihar etti
32                                   Tabi
1000               Seçim heyecanı yaşıyor
209                Uğramaması ayıp olurdu
1546                      Yıl Cami burası
728                    Ne kadar istiyoruz
390                        Ruhu şad olsun
482                    Ne oldu bilmiyorum
875               Söz veriyorum çözeceğim
749                       Gençler oynuyor
946                  Hep beraber olacağız
1517    Normali, siyasilerin konuşmasıdır
445                        Neden bu düzen
997                      İttifaklar vardı
929                                     4
179                                  Öyle
912                                  Asla
497     Milletimizle beraber kurtaracağız
635                       Bunları anlatın
1121                           Yapıyorlar
1255                 İstanbul’a so